<a href="https://colab.research.google.com/github/MohamedIlzam/SE3082_Lab10_IT24610792/blob/main/SE3082_Lab10_IT24610792.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SE3082 – Parallel Computing Lab 10
## CUDA Programming Using Google Colab
### GPU Environment Setup

## Exercise 1 – Timing a GPU Program
### Part (a) – Timing Vector Addition

In [1]:
!nvidia-smi
!nvidia-smi --query-gpu=name,compute_cap --format=csv

Fri Oct  9 18:55:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              8W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:

%%writefile ex1_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 512

__global__ void add(int *a, int *b, int *c) {
    // Each block performs one addition
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

// Track CPU wall-clock time in seconds
double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    int size = N * sizeof(int);
    double dStart, dElaps;

    // Allocate GPU memory
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate CPU memory and generate input values
    a = (int *)malloc(size);
    random_ints(a, N);

    b = (int *)malloc(size);
    random_ints(b, N);

    c = (int *)malloc(size);

    // Copy input arrays from CPU to GPU
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // Start timer
    dStart = cpuSecond();


 // Launch CUDA kernel: 512 blocks, 1 thread each
 add<<<N, 1>>>(d_a, d_b, d_c);

 // Wait for GPU to finish before stopping the timer
 cudaDeviceSynchronize();

 // Stop timer
 dElaps = cpuSecond() - dStart;


    // Copy result back to CPU
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    // Print first five results
    for (int r = 0; r < 5; r++)
        printf("%d + %d = %d\n", a[r], b[r], c[r]);

    // Verify all 512 results
    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r])
            errors++;

    printf("Verification: %d mismatches out of %d\n", errors, N);

    // Print GPU kernel time
    printf("Time taken - %.7f\n", dElaps);

    // Cleanup
    free(a);
    free(b);
    free(c);

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}


Overwriting ex1_timed.cu


In [7]:
!nvcc -arch=sm_75 ex1_timed.cu -o ex1_timed && ./ex1_timed

83 + 77 = 160
86 + 34 = 120
77 + 90 = 167
15 + 26 = 41
93 + 24 = 117
Verification: 0 mismatches out of 512
Time taken - 0.0000989


### Synchronization Experiment

**With synchronization:** 0.0006571 seconds

**Without synchronization:** 0.0001230 seconds

**Observation:**
Removing cudaDeviceSynchronize() reduced the measured time because the CPU did not wait for the GPU to finish before stopping the timer.

The calculations still produced 0 mismatches because the subsequent synchronous cudaMemcpy operation waited for the GPU results.

**Conclusion:**
cudaDeviceSynchronize() is necessary for correctly measuring GPU kernel execution time using a CPU timer.

In [8]:

%%writefile ex3_blocks_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 512

// CUDA kernel: one thread per block
__global__ void add(int *a, int *b, int *c) {
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

// Generate random numbers
void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

// CPU wall-clock timer
double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;

    int size = N * sizeof(int);
    double dStart, dElaps;

    // Allocate GPU memory
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate CPU memory
    a = (int *)malloc(size);
    b = (int *)malloc(size);
    c = (int *)malloc(size);

    random_ints(a, N);
    random_ints(b, N);

    // Copy inputs to GPU
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // Start timer
    dStart = cpuSecond();

    // Launch 512 blocks with 1 thread each
    add<<<N, 1>>>(d_a, d_b, d_c);

    // Wait for GPU to finish
    cudaDeviceSynchronize();

    // Stop timer
    dElaps = cpuSecond() - dStart;

    // Copy results to CPU
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    // Print first 5 results
    for (int i = 0; i < 5; i++) {
        printf("%d + %d = %d\n", a[i], b[i], c[i]);
    }

    // Verify all calculations
    int errors = 0;

    for (int i = 0; i < N; i++) {
        if (c[i] != a[i] + b[i]) {
            errors++;
        }
    }

    printf("Verification: %d mismatches out of %d\n",
           errors, N);

    printf("Time taken - %.7f\n", dElaps);

    // Free memory
    free(a);
    free(b);
    free(c);

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}


Writing ex3_blocks_timed.cu


In [9]:
!nvcc -arch=sm_75 ex3_blocks_timed.cu -o ex3_blocks_timed && echo "===== RUN 1 =====" && ./ex3_blocks_timed && echo "===== RUN 2 =====" && ./ex3_blocks_timed && echo "===== RUN 3 =====" && ./ex3_blocks_timed

===== RUN 1 =====
83 + 77 = 160
86 + 34 = 120
77 + 90 = 167
15 + 26 = 41
93 + 24 = 117
Verification: 0 mismatches out of 512
Time taken - 0.0000961
===== RUN 2 =====
83 + 77 = 160
86 + 34 = 120
77 + 90 = 167
15 + 26 = 41
93 + 24 = 117
Verification: 0 mismatches out of 512
Time taken - 0.0000951
===== RUN 3 =====
83 + 77 = 160
86 + 34 = 120
77 + 90 = 167
15 + 26 = 41
93 + 24 = 117
Verification: 0 mismatches out of 512
Time taken - 0.0001521


In [10]:

%%writefile ex4_threads_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 512

// CUDA kernel: one block containing 512 threads
__global__ void addT(int *a, int *b, int *c) {
    c[threadIdx.x] = a[threadIdx.x] + b[threadIdx.x];
}

// Generate random integers
void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++) {
        x[i] = rand() % 100;
    }
}

// CPU wall-clock timer
double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;

    int size = N * sizeof(int);
    double dStart, dElaps;

    // Allocate GPU memory
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate CPU memory
    a = (int *)malloc(size);
    b = (int *)malloc(size);
    c = (int *)malloc(size);

    // Generate random values
    random_ints(a, N);
    random_ints(b, N);

    // Copy inputs to GPU
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // Start timer
    dStart = cpuSecond();

    // Launch 1 block containing 512 threads
    addT<<<1, N>>>(d_a, d_b, d_c);

    // Check kernel launch
    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess) {
        printf("Kernel launch failed: %s\n",
               cudaGetErrorString(err));
        return 1;
    }

    // Wait until GPU finishes
    err = cudaDeviceSynchronize();
    if (err != cudaSuccess) {
        printf("Kernel execution failed: %s\n",
               cudaGetErrorString(err));
        return 1;
    }

    // Stop timer
    dElaps = cpuSecond() - dStart;

    // Copy results back to CPU
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    // Print first 5 results
    for (int i = 0; i < 5; i++) {
        printf("%d) %d + %d = %d\n",
               i, a[i], b[i], c[i]);
    }

    // Verify all 512 additions
    int errors = 0;

    for (int i = 0; i < N; i++) {
        if (c[i] != a[i] + b[i]) {
            errors++;
        }
    }

    printf("Verification: %d mismatches out of %d\n",
           errors, N);

    // Print kernel execution time
    printf("Time taken - %.7f\n", dElaps);

    // Free CPU memory
    free(a);
    free(b);
    free(c);

    // Free GPU memory
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}


Writing ex4_threads_timed.cu


In [11]:
!nvcc -arch=sm_75 ex4_threads_timed.cu -o ex4_threads_timed && echo "===== RUN 1 =====" && ./ex4_threads_timed && echo "===== RUN 2 =====" && ./ex4_threads_timed && echo "===== RUN 3 =====" && ./ex4_threads_timed

===== RUN 1 =====
0) 83 + 77 = 160
1) 86 + 34 = 120
2) 77 + 90 = 167
3) 15 + 26 = 41
4) 93 + 24 = 117
Verification: 0 mismatches out of 512
Time taken - 0.0000989
===== RUN 2 =====
0) 83 + 77 = 160
1) 86 + 34 = 120
2) 77 + 90 = 167
3) 15 + 26 = 41
4) 93 + 24 = 117
Verification: 0 mismatches out of 512
Time taken - 0.0001092
===== RUN 3 =====
0) 83 + 77 = 160
1) 86 + 34 = 120
2) 77 + 90 = 167
3) 15 + 26 = 41
4) 93 + 24 = 117
Verification: 0 mismatches out of 512
Time taken - 0.0001152


#### Lab 09 Exercise 5 – Vector Multiplication (10 Million Elements)

In [12]:

%%writefile ex5_vecmul_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 10000000
#define THREADS_PER_BLOCK 512

// CUDA kernel: element-wise multiplication
__global__ void vecMul(int *a, int *b, int *c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < n) {
        c[i] = a[i] * b[i];
    }
}

// Generate random values
void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++) {
        x[i] = rand() % 100;
    }
}

// CPU wall-clock timer
double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;

    size_t size = (size_t)N * sizeof(int);
    double dStart, dElaps;

    // Allocate GPU memory
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate CPU memory
    a = (int *)malloc(size);
    b = (int *)malloc(size);
    c = (int *)malloc(size);

    // Generate input values
    random_ints(a, N);
    random_ints(b, N);

    // Copy inputs to GPU
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // Calculate blocks required
    int blocks = (N + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;

    printf("Launching %d blocks x %d threads\n",
           blocks, THREADS_PER_BLOCK);

    // Start GPU kernel timer
    dStart = cpuSecond();

    // Launch vector multiplication kernel
    vecMul<<<blocks, THREADS_PER_BLOCK>>>(d_a, d_b, d_c, N);

    // Check kernel launch
    cudaError_t err = cudaGetLastError();

    if (err != cudaSuccess) {
        printf("Kernel launch failed: %s\n",
               cudaGetErrorString(err));
        return 1;
    }

    // Wait until the GPU finishes
    err = cudaDeviceSynchronize();

    if (err != cudaSuccess) {
        printf("Kernel execution failed: %s\n",
               cudaGetErrorString(err));
        return 1;
    }

    // Stop timer
    dElaps = cpuSecond() - dStart;

    // Copy results to CPU
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    // Print last 1000 results
    for (int i = N - 1000; i < N; i++) {
        printf("%d) %d x %d = %d\n",
               i, a[i], b[i], c[i]);
    }

    // Verify all elements
    int errors = 0;

    for (int i = 0; i < N; i++) {
        if (c[i] != a[i] * b[i]) {
            errors++;
        }
    }

    printf("Verification: %d mismatches out of %d\n",
           errors, N);

    // Print kernel-only execution time
    printf("Time taken - %.7f\n", dElaps);

    // Cleanup
    free(a);
    free(b);
    free(c);

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}


Writing ex5_vecmul_timed.cu


In [13]:
%%bash
set -euo pipefail

nvcc -arch=sm_75 ex5_vecmul_timed.cu -o ex5_vecmul_timed

for run in 1 2 3; do
    echo "===== RUN $run ====="
    ./ex5_vecmul_timed | grep -E '^Launching|^9999000\)|^9999999\)|^Verification:|^Time taken'
done

===== RUN 1 =====
Launching 19532 blocks x 512 threads
9999000) 76 x 52 = 3952
9999999) 31 x 73 = 2263
Verification: 0 mismatches out of 10000000
Time taken - 0.0006080
===== RUN 2 =====
Launching 19532 blocks x 512 threads
9999000) 76 x 52 = 3952
9999999) 31 x 73 = 2263
Verification: 0 mismatches out of 10000000
Time taken - 0.0006020
===== RUN 3 =====
Launching 19532 blocks x 512 threads
9999000) 76 x 52 = 3952
9999999) 31 x 73 = 2263
Verification: 0 mismatches out of 10000000
Time taken - 0.0006402


#### Lab 09 Exercise 6 – Matrix Multiplication (100 Million Elements)

In [14]:

%%writefile ex6_matrix_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define ROWS 10000
#define COLS 10000

// CUDA kernel: element-wise matrix multiplication
__global__ void mulElem(int *a, int *b, int *c,
                        int rows, int cols) {

    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < rows && col < cols) {
        size_t idx = (size_t)row * cols + col;
        c[idx] = a[idx] * b[idx];
    }
}

// Generate random integers
void random_ints(int *x, size_t count) {
    for (size_t i = 0; i < count; i++) {
        x[i] = rand() % 100;
    }
}

// CPU wall-clock timer
double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;

    size_t count = (size_t)ROWS * COLS;
    size_t size = count * sizeof(int);

    double dStart, dElaps;

    // Allocate GPU memory
    cudaError_t err;
    err = cudaMalloc((void **)&d_a, size);
    if (err != cudaSuccess) {
        printf("GPU allocation failed: %s\n",
               cudaGetErrorString(err));
        return 1;
    }

    err = cudaMalloc((void **)&d_b, size);
    if (err != cudaSuccess) {
        printf("GPU allocation failed: %s\n",
               cudaGetErrorString(err));
        return 1;
    }

    err = cudaMalloc((void **)&d_c, size);
    if (err != cudaSuccess) {
        printf("GPU allocation failed: %s\n",
               cudaGetErrorString(err));
        return 1;
    }

    // Allocate CPU memory
    a = (int *)malloc(size);
    b = (int *)malloc(size);
    c = (int *)malloc(size);

    if (!a || !b || !c) {
        printf("CPU memory allocation failed\n");
        return 1;
    }

    // Generate input values
    random_ints(a, count);
    random_ints(b, count);

    // Copy inputs to GPU
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // Configure 2D threads and blocks
    dim3 threadsPerBlock(32, 16);

    dim3 numBlocks(
        (COLS + threadsPerBlock.x - 1) / threadsPerBlock.x,
        (ROWS + threadsPerBlock.y - 1) / threadsPerBlock.y
    );

    printf("Grid: %d x %d blocks\n",
           numBlocks.x, numBlocks.y);

    // Start kernel timer
    dStart = cpuSecond();

    // Launch matrix multiplication kernel
    mulElem<<<numBlocks, threadsPerBlock>>>(
        d_a, d_b, d_c, ROWS, COLS
    );

    // Check kernel launch
    err = cudaGetLastError();

    if (err != cudaSuccess) {
        printf("Kernel launch failed: %s\n",
               cudaGetErrorString(err));
        return 1;
    }

    // Wait for GPU to finish
    err = cudaDeviceSynchronize();

    if (err != cudaSuccess) {
        printf("Kernel execution failed: %s\n",
               cudaGetErrorString(err));
        return 1;
    }

    // Stop kernel timer
    dElaps = cpuSecond() - dStart;

    // Copy results back to CPU
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    // Print last 1000 results
    for (int col = COLS - 1000; col < COLS; col++) {
        int row = ROWS - 1;
        size_t idx = (size_t)row * COLS + col;

        printf("C[%d][%d]) %d x %d = %d\n",
               row, col, a[idx], b[idx], c[idx]);
    }

    // Verify all 100 million elements
    size_t errors = 0;

    for (size_t i = 0; i < count; i++) {
        if (c[i] != a[i] * b[i]) {
            errors++;
        }
    }

    printf("Verification: %zu mismatches out of %zu\n",
           errors, count);

    // Print kernel-only execution time
    printf("Time taken - %.7f\n", dElaps);

    // Cleanup
    free(a);
    free(b);
    free(c);

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}


Writing ex6_matrix_timed.cu


In [15]:
%%bash
set -euo pipefail

nvcc -arch=sm_75 ex6_matrix_timed.cu -o ex6_matrix_timed

for run in 1 2 3; do
    echo "===== RUN $run ====="
    ./ex6_matrix_timed | grep -E '^Grid:|^C\[9999\]\[9000\]|^C\[9999\]\[9999\]|^Verification:|^Time taken'
done

===== RUN 1 =====
Grid: 313 x 625 blocks
C[9999][9000]) 29 x 56 = 1624
C[9999][9999]) 86 x 21 = 1806
Verification: 0 mismatches out of 100000000
Time taken - 0.0051181
===== RUN 2 =====
Grid: 313 x 625 blocks
C[9999][9000]) 29 x 56 = 1624
C[9999][9999]) 86 x 21 = 1806
Verification: 0 mismatches out of 100000000
Time taken - 0.0050220
===== RUN 3 =====
Grid: 313 x 625 blocks
C[9999][9000]) 29 x 56 = 1624
C[9999][9999]) 86 x 21 = 1806
Verification: 0 mismatches out of 100000000
Time taken - 0.0050290


### Part (b) – GPU Kernel Timing Results

| Lab 09 Program | Run 1 (s) | Run 2 (s) | Run 3 (s) |
|---|---:|---:|---:|
| Exercise 3 – 512 blocks × 1 thread | 0.0000961 | 0.0000951 | 0.0001521 |
| Exercise 4 – 1 block × 512 threads | 0.0000989 | 0.0001092 | 0.0001152 |
| Exercise 5 – 10 million elements | 0.0006080 | 0.0006020 | 0.0006402 |
| Exercise 6 – 100 million elements | 0.0051181 | 0.0050220 | 0.0050290 |

**Observations:**

Exercises 3 and 4 had similar execution times. At this small workload, kernel launch overhead and timing variations affect the measurements.

Exercise 6 processed 10 times more elements than Exercise 5, but its execution time was approximately 8 times longer. GPU scheduling, memory access, and fixed overhead can affect how execution time scales.

### Part (c) – Kernel Time Versus Whole-Sequence Time

In [16]:

%%writefile ex5_vecmul_100m_compare.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 100000000
#define THREADS_PER_BLOCK 512

// GPU kernel: element-wise vector multiplication
__global__ void vecMul(int *a, int *b, int *c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < n) {
        c[i] = a[i] * b[i];
    }
}

// Generate random integers
void random_ints(int *x, size_t count) {
    for (size_t i = 0; i < count; i++) {
        x[i] = rand() % 100;
    }
}

// CPU wall-clock timer
double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

// Check CUDA operations
void checkCuda(cudaError_t err, const char *step) {
    if (err != cudaSuccess) {
        printf("CUDA error at %s: %s\n",
               step, cudaGetErrorString(err));
        exit(1);
    }
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;

    size_t count = (size_t)N;
    size_t size = count * sizeof(int);

    double kernelStart, kernelTime;
    double wholeStart, wholeTime;

    // Allocate GPU memory
    checkCuda(cudaMalloc((void **)&d_a, size), "cudaMalloc d_a");
    checkCuda(cudaMalloc((void **)&d_b, size), "cudaMalloc d_b");
    checkCuda(cudaMalloc((void **)&d_c, size), "cudaMalloc d_c");

    // Allocate CPU memory
    a = (int *)malloc(size);
    b = (int *)malloc(size);
    c = (int *)malloc(size);

    if (!a || !b || !c) {
        printf("CPU memory allocation failed\n");
        return 1;
    }

    // Generate 100 million input values per vector
    random_ints(a, count);
    random_ints(b, count);

    // Calculate required blocks
    int blocks = (N + THREADS_PER_BLOCK - 1)
                 / THREADS_PER_BLOCK;

    printf("Launching %d blocks x %d threads\n",
           blocks, THREADS_PER_BLOCK);

    // ========================================
    // WHOLE SEQUENCE TIMER STARTS HERE
    // ========================================
    wholeStart = cpuSecond();

    // Copy input vectors from CPU to GPU
    checkCuda(
        cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice),
        "Copy A to GPU"
    );

    checkCuda(
        cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice),
        "Copy B to GPU"
    );

    // ========================================
    // KERNEL-ONLY TIMER STARTS HERE
    // ========================================
    kernelStart = cpuSecond();

    vecMul<<<blocks, THREADS_PER_BLOCK>>>(
        d_a, d_b, d_c, N
    );

    checkCuda(cudaGetLastError(), "Kernel launch");
    checkCuda(cudaDeviceSynchronize(), "Kernel execution");

    // Stop kernel-only timer
    kernelTime = cpuSecond() - kernelStart;

    // Copy output vector from GPU to CPU
    checkCuda(
        cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost),
        "Copy result to CPU"
    );

    // ========================================
    // WHOLE SEQUENCE TIMER ENDS HERE
    // ========================================
    wholeTime = cpuSecond() - wholeStart;

    // Print last 1000 results
    for (int i = N - 1000; i < N; i++) {
        printf("%d) %d x %d = %d\n",
               i, a[i], b[i], c[i]);
    }

    // Verify all 100 million results on CPU
    size_t errors = 0;

    for (size_t i = 0; i < count; i++) {
        if (c[i] != a[i] * b[i]) {
            errors++;
        }
    }

    printf("Verification: %zu mismatches out of %zu\n",
           errors, count);

    // Display both measured times
    printf("Kernel only time - %.7f seconds\n", kernelTime);
    printf("Whole sequence time - %.7f seconds\n", wholeTime);

    // Cleanup
    free(a);
    free(b);
    free(c);

    checkCuda(cudaFree(d_a), "Free d_a");
    checkCuda(cudaFree(d_b), "Free d_b");
    checkCuda(cudaFree(d_c), "Free d_c");

    return 0;
}


Writing ex5_vecmul_100m_compare.cu


In [17]:
%%bash
set -euo pipefail

nvcc -arch=sm_75 ex5_vecmul_100m_compare.cu -o ex5_vecmul_100m_compare

./ex5_vecmul_100m_compare | grep -E '^Launching|^99999000\)|^99999999\)|^Verification:|^Kernel only time|^Whole sequence time|^CUDA error|failed'

Launching 195313 blocks x 512 threads
99999000) 29 x 56 = 1624
99999999) 86 x 21 = 1806
Verification: 0 mismatches out of 100000000
Kernel only time - 0.0047629 seconds
Whole sequence time - 0.4440050 seconds


### Exercise 1(c) – Timing Comparison Results

- Kernel-only time: 0.0047629 seconds
- Whole-sequence time: 0.4440050 seconds

**Observation:**

The whole sequence took approximately 93 times longer than the kernel-only computation.

**Conclusion:**

Data transfers between CPU and GPU memory introduce significant overhead. GPU computation is most beneficial when enough processing is performed to justify the cost of transferring data.

## Exercise 2 – Catching CUDA Errors Using the CHECK() Macro

In [26]:

%%writefile ex2_check.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 512

// CUDA error checking macro
#define CHECK(call)                                                   \
    do {                                                              \
        const cudaError_t error = (call);                             \
        if (error != cudaSuccess) {                                   \
            printf("Error: %s:%d, ", __FILE__, __LINE__);             \
            printf("code:%d, reason: %s\n", error,                    \
                   cudaGetErrorString(error));                        \
            exit(1);                                                  \
        }                                                             \
    } while (0)

// CUDA kernel: vector addition
__global__ void add(int *a, int *b, int *c) {
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

// Generate random numbers
void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++) {
        x[i] = rand() % 100;
    }
}

// CPU wall-clock timer
double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;

    int size = N * sizeof(int);
    double dStart, dElaps;

    // TODO 1: Check all three CUDA allocations
    CHECK(cudaMalloc((void **)&d_a, size));
    CHECK(cudaMalloc((void **)&d_b, size));
    CHECK(cudaMalloc((void **)&d_c, size));

    // Allocate host memory
    a = (int *)malloc(size);
    random_ints(a, N);

    b = (int *)malloc(size);
    random_ints(b, N);

    c = (int *)malloc(size);

    // TODO 2: Check host-to-device copies
    CHECK(cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice));

    // Start timer
    dStart = cpuSecond();

    // Correct launch: 512 blocks, 1 thread each
    add<<<N, 1>>>(d_a, d_b, d_c);

    // TODO 3: Check launch and synchronization
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());

    // Stop timer
    dElaps = cpuSecond() - dStart;

    // TODO 4: Check device-to-host copy
    CHECK(cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost));

    // Print first 5 results
    for (int r = 0; r < 5; r++) {
        printf("%d + %d = %d\n", a[r], b[r], c[r]);
    }

    // Verify all results
    int errors = 0;

    for (int r = 0; r < N; r++) {
        if (c[r] != a[r] + b[r]) {
            errors++;
        }
    }

    printf("Verification: %d mismatches out of %d\n",
           errors, N);

    printf("Time taken - %.7f\n", dElaps);

    // Free CPU memory
    free(a);
    free(b);
    free(c);

    // TODO 5: Check CUDA memory releases
    CHECK(cudaFree(d_a));
    CHECK(cudaFree(d_b));
    CHECK(cudaFree(d_c));

    return 0;
}


Overwriting ex2_check.cu


In [27]:
!nvcc -arch=sm_75 ex2_check.cu -o ex2_check && ./ex2_check

83 + 77 = 160
86 + 34 = 120
77 + 90 = 167
15 + 26 = 41
93 + 24 = 117
Verification: 0 mismatches out of 512
Time taken - 0.0001061


In [28]:

%%writefile ex2_no_check_experiment.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 512

// GPU kernel from Exercise 1
__global__ void add(int *a, int *b, int *c) {
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++) {
        x[i] = rand() % 100;
    }
}

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;

    int size = N * sizeof(int);
    double dStart, dElaps;

    // Allocate GPU memory without CHECK
    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    // Allocate CPU memory
    a = (int *)malloc(size);
    random_ints(a, N);

    b = (int *)malloc(size);
    random_ints(b, N);

    c = (int *)malloc(size);

    // Copy inputs to GPU
    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    // Initialize GPU results to zero for the experiment
    cudaMemset(d_c, 0, size);

    dStart = cpuSecond();

    // EXPERIMENT C:
    // Invalid configuration - 2048 threads per block
    add<<<N, 2048>>>(d_a, d_b, d_c);

    // NO CHECK(cudaGetLastError()) here
    // Synchronization error is also not checked
    cudaDeviceSynchronize();

    dElaps = cpuSecond() - dStart;

    // Copy results back to CPU
    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    // Print first five results
    for (int r = 0; r < 5; r++) {
        printf("%d + %d = %d\n", a[r], b[r], c[r]);
    }

    // Verify all results
    int errors = 0;

    for (int r = 0; r < N; r++) {
        if (c[r] != a[r] + b[r]) {
            errors++;
        }
    }

    printf("Verification: %d mismatches out of %d\n",
           errors, N);

    printf("Time taken - %.7f\n", dElaps);

    // Cleanup
    free(a);
    free(b);
    free(c);

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}


Overwriting ex2_no_check_experiment.cu


In [29]:
!nvcc -arch=sm_75 ex2_no_check_experiment.cu -o ex2_no_check_experiment && ./ex2_no_check_experiment

83 + 77 = 0
86 + 34 = 0
77 + 90 = 0
15 + 26 = 0
93 + 24 = 0
Verification: 512 mismatches out of 512
Time taken - 0.0001650


### Exercise 2 – Error Detection Experiments

**Experiment A – Too Many Threads**

- Launched 2048 threads per block.
- Observed: code 1, reason: invalid argument.
- The CHECK macro detected the invalid kernel launch and terminated the program.
- The lab sheet expected code 9, but the installed CUDA environment returned code 1.

**Experiment B – Impossible Memory Allocation**

- Requested 1 petabyte of GPU memory.
- Observed: code 2, reason: out of memory.
- The CHECK macro detected the failed allocation and stopped execution.

**Experiment C – Without CHECK**

- Launched 2048 threads per block without checking CUDA errors.
- Observed: 512 mismatches out of 512.
- The kernel failed, but the program continued without reporting the CUDA error.
- Verification was necessary to detect the incorrect results.

**Conclusion**

The CHECK macro helps identify CUDA errors immediately. Checking kernel launches, memory operations and synchronization prevents silent failures.

After the experiments, the original program was restored to its correct configuration.

## Exercise 3 – Scaling a Picture with a 2D Grid

In [30]:

%%writefile ex3_picture.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define WIDTH 800
#define HEIGHT 600

// CUDA error checking macro
#define CHECK(call)                                                   \
    do {                                                              \
        const cudaError_t error = (call);                             \
        if (error != cudaSuccess) {                                   \
            printf("Error: %s:%d, ", __FILE__, __LINE__);             \
            printf("code:%d, reason: %s\n", error,                    \
                   cudaGetErrorString(error));                        \
            exit(1);                                                  \
        }                                                             \
    } while (0)

// CPU wall-clock timer
double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

// Scale each pixel by 2, capped at 255
__global__ void PictureKernel(
    unsigned char *d_Pin,
    unsigned char *d_Pout,
    int height,
    int width
) {
    int Row = blockIdx.y * blockDim.y + threadIdx.y;
    int Col = blockIdx.x * blockDim.x + threadIdx.x;

    if ((Row < height) && (Col < width)) {
        int v = 2 * d_Pin[Row * width + Col];

        d_Pout[Row * width + Col] =
            (v > 255) ? 255 : v;
    }
}

int main(void) {
    size_t count = (size_t)WIDTH * HEIGHT;
    size_t size = count * sizeof(unsigned char);

    unsigned char *h_in, *h_out;
    unsigned char *d_in, *d_out;

    double dStart, dElaps;

    // Allocate CPU memory
    h_in = (unsigned char *)malloc(size);
    h_out = (unsigned char *)malloc(size);

    if (!h_in || !h_out) {
        printf("CPU memory allocation failed\n");
        return 1;
    }

    // Allocate GPU memory
    CHECK(cudaMalloc((void **)&d_in, size));
    CHECK(cudaMalloc((void **)&d_out, size));

    // TODO 1: Generate random pixel values (0 to 255)
    for (size_t i = 0; i < count; i++) {
        h_in[i] = rand() % 256;
    }

    // Copy original picture to GPU
    CHECK(cudaMemcpy(
        d_in, h_in, size, cudaMemcpyHostToDevice
    ));

    // TODO 2: 16 x 16 threads per block
    dim3 threadsPerBlock(16, 16);

    // TODO 3: Calculate the 2D grid size
    dim3 numBlocks(
        (WIDTH + threadsPerBlock.x - 1) / threadsPerBlock.x,
        (HEIGHT + threadsPerBlock.y - 1) / threadsPerBlock.y
    );

    printf("Grid: %d x %d blocks of %d x %d threads\n",
           numBlocks.x, numBlocks.y,
           threadsPerBlock.x, threadsPerBlock.y);

    // Start GPU kernel timer
    dStart = cpuSecond();

    // TODO 4: Launch PictureKernel
    PictureKernel<<<numBlocks, threadsPerBlock>>>(
        d_in, d_out, HEIGHT, WIDTH
    );

    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());

    // Stop timer
    dElaps = cpuSecond() - dStart;

    // Copy processed picture back to CPU
    CHECK(cudaMemcpy(
        h_out, d_out, size, cudaMemcpyDeviceToHost
    ));

    // TODO 5: Display top-left 50 x 50 original pixels
    printf("\nOriginal Picture (50 x 50):\n");

    for (int row = 0; row < 50; row++) {
        for (int col = 0; col < 50; col++) {
            int idx = row * WIDTH + col;
            printf("%3d ", (int)h_in[idx]);
        }
        printf("\n");
    }

    // Display top-left 50 x 50 updated pixels
    printf("\nUpdated Picture (50 x 50):\n");

    for (int row = 0; row < 50; row++) {
        for (int col = 0; col < 50; col++) {
            int idx = row * WIDTH + col;
            printf("%3d ", (int)h_out[idx]);
        }
        printf("\n");
    }

    // TODO 6: Verify all pixels and count capped pixels
    size_t errors = 0;
    size_t capped = 0;

    for (size_t i = 0; i < count; i++) {
        int original = (int)h_in[i];
        int expected = 2 * original;

        if (expected > 255) {
            expected = 255;
            capped++;
        }

        if ((int)h_out[i] != expected) {
            errors++;
        }
    }

    printf("\nVerification: %zu mismatches out of %zu\n",
           errors, count);

    printf("Pixels capped at 255: %zu\n", capped);

    printf("Time taken - %.7f\n", dElaps);

    // Cleanup
    free(h_in);
    free(h_out);

    CHECK(cudaFree(d_in));
    CHECK(cudaFree(d_out));

    return 0;
}


Writing ex3_picture.cu


In [31]:
!nvcc -arch=sm_75 ex3_picture.cu -o ex3_picture && ./ex3_picture

Grid: 50 x 38 blocks of 16 x 16 threads

Original Picture (50 x 50):
103 198 105 115  81 255  74 236  41 205 186 171 242 251 227  70 124 194  84 248  27 232 231 141 118  90  46  99  51 159 201 154 102  50  13 183  49  88 163  90  37  93   5  23  88 233  94 212 171 178 
 96  34  10 167  77  49 181  91   3 160  13  34  13  71  93 205 155 135 120  86 213 112  76 156 134 234  15 152 242 235 156  83  13 167 250  90 216 176 181 219  80 194 253  93   9  90  42 165 226 163 
210 229 220  71  16 152 132 214 161  59  36  81  31  29 107 245  90 125  16 216  23 252 165  61 140  36 239 252 218 206  78 172 179  42 243 196 195 119 154 100 178 190 181 209 219  32 198  53 157 214 
158 212 187 189 162 172 238 192 141 116  54 141  24 225  34 225 154   4  34 178 109 178 216 130 145 231 176 222 132 115 155  34  71  86 223 233   2 205 169 143  65 224  28  90 193  63  59  91  67  93 
219 240 232   2 175 227 102  92  27 167 209 145 153  42 245 250 103 153 124 186 196 109  26  59 117 143  79  87 135 187  33  98

## Exercise 4 – Multiplying Two 3D Matrices Element by Element

This exercise uses a three-dimensional CUDA grid to multiply corresponding elements of two 500 × 500 × 500 matrices.

Each GPU thread calculates one element. The program verifies all 125 million results and measures kernel execution time.

In [32]:

%%writefile ex4_volume.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define NX 500
#define NY 500
#define NZ 500

// CUDA error checking macro
#define CHECK(call)                                                   \
    do {                                                              \
        const cudaError_t error = (call);                             \
        if (error != cudaSuccess) {                                   \
            printf("Error: %s:%d, ", __FILE__, __LINE__);             \
            printf("code:%d, reason: %s\n", error,                    \
                   cudaGetErrorString(error));                        \
            exit(1);                                                  \
        }                                                             \
    } while (0)

// CPU wall-clock timer
double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

// 3D element-wise multiplication kernel
__global__ void mulElem3D(
    int *a, int *b, int *c,
    int nx, int ny, int nz
) {
    // TODO 1: Calculate the 3D coordinates
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    int z = blockIdx.z * blockDim.z + threadIdx.z;

    // TODO 2: Check all three boundaries
    if (x < nx && y < ny && z < nz) {

        // TODO 3: Convert 3D coordinates to flat index
        size_t idx = ((size_t)z * ny + y) * nx + x;

        c[idx] = a[idx] * b[idx];
    }
}

// Generate random input values
void random_ints(int *x, size_t count) {
    for (size_t i = 0; i < count; i++) {
        x[i] = rand() % 100;
    }
}

int main(void) {
    // Total: 125 million elements
    size_t count = (size_t)NX * NY * NZ;
    size_t size = count * sizeof(int);

    int *a, *b, *c;
    int *d_a, *d_b, *d_c;

    double dStart, dElaps;

    // Allocate GPU memory
    CHECK(cudaMalloc((void **)&d_a, size));
    CHECK(cudaMalloc((void **)&d_b, size));
    CHECK(cudaMalloc((void **)&d_c, size));

    // Allocate CPU memory
    a = (int *)malloc(size);
    b = (int *)malloc(size);
    c = (int *)malloc(size);

    if (!a || !b || !c) {
        printf("CPU memory allocation failed\n");
        return 1;
    }

    // Generate random data
    random_ints(a, count);
    random_ints(b, count);

    // Copy inputs from CPU to GPU
    CHECK(cudaMemcpy(
        d_a, a, size, cudaMemcpyHostToDevice
    ));

    CHECK(cudaMemcpy(
        d_b, b, size, cudaMemcpyHostToDevice
    ));

    // 8 x 8 x 8 = 512 threads per block
    dim3 threadsPerBlock(8, 8, 8);

    // TODO 4: Calculate 3D grid dimensions
    dim3 numBlocks(
        (NX + threadsPerBlock.x - 1) / threadsPerBlock.x,
        (NY + threadsPerBlock.y - 1) / threadsPerBlock.y,
        (NZ + threadsPerBlock.z - 1) / threadsPerBlock.z
    );

    printf("Grid: %d x %d x %d blocks\n",
           numBlocks.x, numBlocks.y, numBlocks.z);

    // Start kernel timer
    dStart = cpuSecond();

    // TODO 5: Launch the 3D CUDA kernel
    mulElem3D<<<numBlocks, threadsPerBlock>>>(
        d_a, d_b, d_c, NX, NY, NZ
    );

    // Check launch and wait for completion
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());

    // Stop kernel timer
    dElaps = cpuSecond() - dStart;

    // Copy GPU results back to CPU
    CHECK(cudaMemcpy(
        c, d_c, size, cudaMemcpyDeviceToHost
    ));

    // TODO 6: Print the last 1000 elements
    printf("\nLast 1000 results:\n");

    for (size_t i = count - 1000; i < count; i++) {

        size_t x = i % NX;
        size_t y = (i / NX) % NY;
        size_t z = i / ((size_t)NX * NY);

        printf("C[%zu][%zu][%zu] = %d x %d = %d\n",
               z, y, x, a[i], b[i], c[i]);
    }

    // TODO 7: Verify all 125 million elements
    size_t errors = 0;

    for (size_t i = 0; i < count; i++) {
        if (c[i] != a[i] * b[i]) {
            errors++;
        }
    }

    printf("\nVerification: %zu mismatches out of %zu\n",
           errors, count);

    // Print execution time
    printf("Time taken - %.7f\n", dElaps);

    // Free CPU memory
    free(a);
    free(b);
    free(c);

    // Free GPU memory
    CHECK(cudaFree(d_a));
    CHECK(cudaFree(d_b));
    CHECK(cudaFree(d_c));

    return 0;
}


Writing ex4_volume.cu


In [33]:
!nvcc -arch=sm_75 ex4_volume.cu -o ex4_volume && ./ex4_volume

Grid: 63 x 63 x 63 blocks

Last 1000 results:
C[499][498][0] = 76 x 15 = 1140
C[499][498][1] = 38 x 93 = 3534
C[499][498][2] = 34 x 11 = 374
C[499][498][3] = 85 x 47 = 3995
C[499][498][4] = 3 x 38 = 114
C[499][498][5] = 43 x 82 = 3526
C[499][498][6] = 16 x 52 = 832
C[499][498][7] = 28 x 23 = 644
C[499][498][8] = 32 x 28 = 896
C[499][498][9] = 80 x 98 = 7840
C[499][498][10] = 69 x 81 = 5589
C[499][498][11] = 45 x 79 = 3555
C[499][498][12] = 30 x 2 = 60
C[499][498][13] = 41 x 13 = 533
C[499][498][14] = 30 x 81 = 2430
C[499][498][15] = 30 x 74 = 2220
C[499][498][16] = 41 x 13 = 533
C[499][498][17] = 14 x 18 = 252
C[499][498][18] = 83 x 20 = 1660
C[499][498][19] = 13 x 28 = 364
C[499][498][20] = 86 x 17 = 1462
C[499][498][21] = 13 x 6 = 78
C[499][498][22] = 88 x 57 = 5016
C[499][498][23] = 71 x 29 = 2059
C[499][498][24] = 28 x 17 = 476
C[499][498][25] = 78 x 2 = 156
C[499][498][26] = 50 x 37 = 1850
C[499][498][27] = 86 x 40 = 3440
C[499][498][28] = 28 x 35 = 980
C[499][498][29] = 15 x 1 = 

### Exercise 4 – 3D Matrix Multiplication Results

**Matrix dimensions:** 500 × 500 × 500

**Grid configuration:** 63 × 63 × 63 blocks

**Threads per block:** 8 × 8 × 8 = 512

**Total elements:** 125,000,000

**Verification:** 0 mismatches out of 125,000,000

**Kernel execution time:** 0.0079250 seconds

### Comparison with Lab 09 Exercise 6

| Program | Elements | Kernel Time |
|---|---:|---:|
| Lab 09 Exercise 6 – 2D | 100,000,000 | 0.0050564 s (average of 3 runs) |
| Lab 10 Exercise 4 – 3D | 125,000,000 | 0.0079250 s |

**Observation:**

The 3D kernel took approximately 1.57 times as long as the 2D kernel, while processing 25% more elements.

**Explanation:**

Both kernels perform element-wise multiplication. The difference in execution time may be affected by the number of elements, memory access patterns, thread arrangement, GPU scheduling, and measurement variation.

**Conclusion:**

CUDA supports both 2D and 3D thread arrangements, making it possible to process large matrices and volumes in parallel. Proper thread indexing and bounds checking are necessary to produce correct results.